In [18]:
import os

for fichier in os.listdir("/home/jovyan/work"):
    print("-", fichier)

- TP3_Spark_MLlib.ipynb
- TP_Spark_Jour2.ipynb
- train.csv
- winequality-red.csv
- .ipynb_checkpoints


In [26]:
# EXERCICE 1 — Spark MLlib
# ÉTAPE 3 — Initialisation de Spark

from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("TP3-Spark-MLlib") \
    .getOrCreate()

# Afficher la version de Spark
print("Version de Spark :", spark.version)

Version de Spark : 3.5.0


In [31]:
# EXERCICE 1 — ÉTAPE 4
# Rechargement du dataset Wine Quality

df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .option("sep", ",") \
    .csv("/home/jovyan/work/winequality-red.csv")

# Afficher les colonnes pour vérifier
print("Colonnes :", df.columns)

# Afficher les premières lignes
df.show(5)

Colonnes : ['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar', 'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density', 'pH', 'sulphates', 'alcohol', 'quality']
+-------------+----------------+-----------+--------------+---------+-------------------+--------------------+-------+----+---------+-------+-------+
|fixed acidity|volatile acidity|citric acid|residual sugar|chlorides|free sulfur dioxide|total sulfur dioxide|density|  pH|sulphates|alcohol|quality|
+-------------+----------------+-----------+--------------+---------+-------------------+--------------------+-------+----+---------+-------+-------+
|          7.4|             0.7|        0.0|           1.9|    0.076|               11.0|                34.0| 0.9978|3.51|     0.56|    9.4|      5|
|          7.8|            0.88|        0.0|           2.6|    0.098|               25.0|                67.0| 0.9968| 3.2|     0.68|    9.8|      5|
|          7.8|            0.76|       0.04|           2

In [32]:
# ÉTAPE 5 — Explorer le dataset

# Afficher le type des colonnes
df.printSchema()

# Afficher le nom des colonnes
print("Colonnes :")
print(df.columns)

# Compter le nombre de ligne
print("Nombre de lignes :", df.count())

root
 |-- fixed acidity: double (nullable = true)
 |-- volatile acidity: double (nullable = true)
 |-- citric acid: double (nullable = true)
 |-- residual sugar: double (nullable = true)
 |-- chlorides: double (nullable = true)
 |-- free sulfur dioxide: double (nullable = true)
 |-- total sulfur dioxide: double (nullable = true)
 |-- density: double (nullable = true)
 |-- pH: double (nullable = true)
 |-- sulphates: double (nullable = true)
 |-- alcohol: double (nullable = true)
 |-- quality: integer (nullable = true)

Colonnes :
['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar', 'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density', 'pH', 'sulphates', 'alcohol', 'quality']
Nombre de lignes : 1599


In [33]:
for colonne in df.columns:
    print(repr(colonne))

'fixed acidity'
'volatile acidity'
'citric acid'
'residual sugar'
'chlorides'
'free sulfur dioxide'
'total sulfur dioxide'
'density'
'pH'
'sulphates'
'alcohol'
'quality'


In [35]:
# ÉTAPE 6 préparation des features

from pyspark.ml.feature import VectorAssembler

# Toutes les colonnes sauf "quality"
feature_cols = [col for col in df.columns if col != "quality"]

print("Variables utilisées :", feature_cols)

# Création de l'assembleur
assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

# Création de la colonne feature
data = assembler.transform(df)

# Afficher les features et la qualité
data.select("features", "quality").show(5)

Variables utilisées : ['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar', 'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density', 'pH', 'sulphates', 'alcohol']
+--------------------+-------+
|            features|quality|
+--------------------+-------+
|[7.4,0.7,0.0,1.9,...|      5|
|[7.8,0.88,0.0,2.6...|      5|
|[7.8,0.76,0.04,2....|      5|
|[11.2,0.28,0.56,1...|      6|
|[7.4,0.7,0.0,1.9,...|      5|
+--------------------+-------+
only showing top 5 rows



In [36]:
# ÉTAPE 7  préparation du label

# Renommer la colonne quality en label
data = data.withColumnRenamed("quality", "label")

# Garder que les colonne utile
data = data.select("features", "label")

# Vérifier la structure finale
data.printSchema()

# Afficher quelques lignes
data.show(5)

root
 |-- features: vector (nullable = true)
 |-- label: integer (nullable = true)

+--------------------+-----+
|            features|label|
+--------------------+-----+
|[7.4,0.7,0.0,1.9,...|    5|
|[7.8,0.88,0.0,2.6...|    5|
|[7.8,0.76,0.04,2....|    5|
|[11.2,0.28,0.56,1...|    6|
|[7.4,0.7,0.0,1.9,...|    5|
+--------------------+-----+
only showing top 5 rows



In [37]:
# ÉTAPE 8  séparation des données

# 70% entraînement et 30% test
train_df, test_df = data.randomSplit(
    [0.7, 0.3],
    seed=42
)

# Afficher le nombre de lignes
print("Nombre de lignes train :", train_df.count())
print("Nombre de lignes test :", test_df.count())

Nombre de lignes train : 1173
Nombre de lignes test : 426


In [ ]:
#Question 

#Pourquoi séparer les données ?
#pour entrainer le modèle sur certaines données et vérifié ses performances sur des données inconnu

#Pourquoi utiliser un seed ?
#pour obtenir la même séparation à chaque fois

#Pourquoi ne doit-on pas entraîner et évaluer le modèle exactement sur les mêmes données ?
#Parcequ'il pourrait mémoriser les données et donner des mauvais résultats sur des nouvelles données

In [39]:
# ÉTAPE 9 création du modèle de régression linéaire

# Import de LinearRegression
from pyspark.ml.regression import LinearRegression

# Création du modèle
lr = LinearRegression(
    featuresCol="features",
    labelCol="label"
)

# Entraînement du modèle
model = lr.fit(train_df)

In [40]:
# ÉTAPE 10 prédictions

# Utilisation du modèle sur les données de test
predictions = model.transform(test_df)

# Afficher les valeurs réelles et prédites
predictions.select(
    "label",
    "prediction"
).show(10)

+-----+------------------+
|label|        prediction|
+-----+------------------+
|    7|6.7945086222196185|
|    6| 5.745905956741173|
|    5| 5.117207991283538|
|    7|6.5592721477111855|
|    5|  5.25634203884241|
|    6| 6.949183990708834|
|    6| 6.949183990708834|
|    7| 6.627836688038679|
|    7| 6.299168940405983|
|    7|6.0515315459029555|
+-----+------------------+
only showing top 10 rows



In [ ]:
#Question

#Quelle est la différence entre label et prediction ?
#le label c'est la vraie valeur et prediction c'est la valeur prédite

#Pourquoi les valeurs prédites peuvent-elles être différentes des valeurs réelles ?
#parce qu'il fait une estimation et qu'il ne peut pas prédire parfaitement toutes les valeurs

In [41]:
# ÉTAPE 11 évaluation du modèle

from pyspark.ml.evaluation import RegressionEvaluator

# Création de l'évaluateur RMSE
evaluator_rmse = RegressionEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="rmse"
)

# Calcul du RMSE
rmse = evaluator_rmse.evaluate(predictions)

print("RMSE :", rmse)

RMSE : 0.6670030973323298


In [42]:
# Évaluation avec R²

evaluator_r2 = RegressionEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="r2"
)

# Calcul du R²
r2 = evaluator_r2.evaluate(predictions)

print("R² :", r2)

R² : 0.3470856805859932


In [ ]:
#Question 

#Que mesure le RMSE ?
#il mesure l'écart entre les valeur réel et les valeurs prédites

#Que signifie une valeur faible du RMSE ?
#que les prédictions sont proches des valeurs réels

#Que mesure le R² ?
#il mesure la capacité à expliquer les variations des données

#Comment interpréter le R² obtenu ?
#plus il est proche de 1 plus le modèle explique bien les données

In [43]:
# ÉTAPE 12 coefficients

# Afficher l'intercept
print("Intercept :", model.intercept)

# Afficher chaque coefficient avec sa variable
print("\nCoefficients :")

for feature, coefficient in zip(feature_cols, model.coefficients):
    print(feature, ":", coefficient)

Intercept : 4.296477856903301

Coefficients :
fixed acidity : 0.024504546923137315
volatile acidity : -1.097105427261691
citric acid : -0.24332033090558414
residual sugar : -0.018968324046269917
chlorides : -1.3762852773410217
free sulfur dioxide : 0.006309240953362335
total sulfur dioxide : -0.0031866947688895127
density : -0.28200889700073484
pH : -0.4397618939184932
sulphates : 0.8192129389619762
alcohol : 0.3058895297878435


In [ ]:
#Question

#Quelle variable possède le coefficient le plus élevé ?
#le sulphates avec 0,819

#Quelle variable possède le coefficient le plus faible ?
#le chlorides avec -1,376

#Quel est le signe de chaque coefficient ?
#Positive : fixed acidity, citric acid, residual sugar, free sulfur dioxide, sulphates, alcohol
#Négative : volatile acidity, chlorides, total sulfur dioxide, density, pH

#Comment interpréter ces coefficients dans le contexte du modèle ?
#il montre l'influence de la variable sur la prédiction, toutes les autres variables sont considérées dans le modèle.

In [44]:
#EXERCICE 2 — Monitoring et optimisation Spark
# ÉTAPE 13 Chargement de train.csv

# Chemin du fichier
train_path = "/home/jovyan/work/train.csv"

# Chargement du fichier CSV
df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv(train_path)

# Afficher les premières lignes
df.show(5)

# Afficher le schéma
df.printSchema()

+---------+---------+-------------------+-------------------+---------------+------------------+------------------+------------------+------------------+------------------+-------------+
|       id|vendor_id|    pickup_datetime|   dropoff_datetime|passenger_count|  pickup_longitude|   pickup_latitude| dropoff_longitude|  dropoff_latitude|store_and_fwd_flag|trip_duration|
+---------+---------+-------------------+-------------------+---------------+------------------+------------------+------------------+------------------+------------------+-------------+
|id2875421|        2|2016-03-14 17:24:55|2016-03-14 17:32:30|              1| -73.9821548461914| 40.76793670654297|-73.96463012695312|40.765602111816406|                 N|          455|
|id2377394|        1|2016-06-12 00:43:35|2016-06-12 00:54:38|              1|-73.98041534423828|40.738563537597656|-73.99948120117188| 40.73115158081055|                 N|          663|
|id3858529|        2|2016-01-19 11:35:24|2016-01-19 12:10:48|    

In [45]:
# ÉTAPE 14 compter les lignes

nombre_lignes = df.count()

print("Nombre de lignes :", nombre_lignes)

Nombre de lignes : 1458644


In [ ]:
#Question 

#Quelle opération a déclenché le Job ?
#count()

#Combien de Jobs sont visibles ?
#1

#Combien de Stages ?
#1

#Combien de Tasks ?
#8

#Pourquoi count() déclenche-t-il réellement l'exécution ?
#Car il est une action Spark et les transformation sont exécutées seulement lorsqu'une action est appelée


In [49]:
# ÉTAPE 15 Observer les partitions

# Afficher le nombre de partitions
print("Nombre de partitions :", df.rdd.getNumPartitions())
# Compter le nombre de lignes dans chaque partition

partition_sizes = df.rdd.mapPartitions(
    lambda partition: [sum(1 for _ in partition)]
).collect()

print("Lignes par partition :", partition_sizes)

Nombre de partitions : 8
Lignes par partition : [186135, 186139, 186139, 186143, 186150, 186149, 186147, 155642]


In [ ]:
#Question

#Qu'est-ce qu'une partition ?
#c'est une partie des données traitée par Spark

#Pourquoi Spark utilise-t-il des partitions ?
#pour traiter les données en parallèle

#Quel lien existe entre partition et Task ?
#une Task traite une partition

#Les données sont-elles réparties uniformément ?
# d'apres les résultats oui : 186135, 186139, 186139, 186143, 186150, 186149, 186147, 155642

In [ ]:
#ÉTAPE 16 Comprendre Job → Stage → Task

#Question

#Qu'est-ce qu'un Job ?
#Un Job est créé lorsqu'une action Spark est exécuté

#Qu'est-ce qu'un Stage ?
#un Stage est une étape d'exécution d'un Job

#Qu'est-ce qu'une Task ?
#une Task est une unité de travail exécutée sur une partition

#Quel lien existe entre les Tasks et les partitions ?
#les Tasks travaille sur les partitions des données

In [50]:
# ÉTAPE 17  Test de repartition()

# Changer le nombre de partitions
df_repartition = df.repartition(8)

# Vérifier le nombre de partitions
print(
    "Nombre de partitions après repartition :",
    df_repartition.rdd.getNumPartitions()
)

print("Nombre de lignes :", df_repartition.count())

Nombre de partitions après repartition : 8
Nombre de lignes : 1458644


In [51]:
# ÉTAPE 18 Test de coalesce()

# Réduire le nombre de partitions
df_coalesce = df_repartition.coalesce(4)

# Vérifier le nombre de partitions
print(
    "Nombre de partitions après coalesce :",
    df_coalesce.rdd.getNumPartitions()
)

print("Nombre de lignes :", df_coalesce.count())

Nombre de partitions après coalesce : 4
Nombre de lignes : 1458644


In [52]:
# ÉTAPE 19 Test du cache

# Mettre le DataFrame en cache
df_cached = df.cache()

print("Première exécution :", df_cached.count())
print("Deuxième exécution :", df_cached.count())

Première exécution : 1458644
Deuxième exécution : 1458644


In [54]:
# ÉTAPE 20 Test de persist()

from pyspark import StorageLevel

# Stocker les données en mémoire
df_persisted = df.persist(
    StorageLevel.MEMORY_AND_DISK
)

print("Nombre de lignes :", df_persisted.count())
print("Niveau de stockage :", df_persisted.storageLevel)
print("Deuxième exécution :", df_persisted.count())

Nombre de lignes : 1458644
Niveau de stockage : Disk Memory Deserialized 1x Replicated
Deuxième exécution : 1458644


In [55]:
# ÉTAPE 21 Libération du cache

# Supprimer le DataFrame du cached
df_cached.unpersist()

# Supprimer le DataFrame persisted
df_persisted.unpersist()


DataFrame[id: string, vendor_id: int, pickup_datetime: timestamp, dropoff_datetime: timestamp, passenger_count: int, pickup_longitude: double, pickup_latitude: double, dropoff_longitude: double, dropoff_latitude: double, store_and_fwd_flag: string, trip_duration: int]